In [1]:
import os
import numpy as np
import scipy.io as sio
from tqdm import tqdm
import h5py

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

import sys
sys.path.append("..") 

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [2]:
from UCI.pre_processing import  load_UCI_dataset, create_dataloaders

In [3]:
X_train, y_train, X_val, y_val, X_test, y_test = load_UCI_dataset(WINDOW_SIZE= 1000, STEP_SIZE= 500)

Total recordings: 12000
Train recordings: 9600
Validation recordings: 1200
Test recordings: 1200


100%|██████████| 9600/9600 [03:40<00:00, 43.56it/s] 


Skipped recordings: 4


100%|██████████| 1200/1200 [00:15<00:00, 79.47it/s]


Skipped recordings: 1


100%|██████████| 1200/1200 [00:14<00:00, 82.41it/s]


Skipped recordings: 0


In [4]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# The shape should be : (batch, channels, length), length = 8*125 = 1000samples, only one channel as PPG and 

X_train: torch.Size([522837, 1, 1000])
y_train: torch.Size([522837, 2])
X_val: torch.Size([66013, 1, 1000])
y_val: torch.Size([66013, 2])
X_test: torch.Size([66296, 1, 1000])
y_test: torch.Size([66296, 2])


In [5]:
print(X_train.dtype)
print(y_train.dtype)

print(torch.isnan(X_train).any())
print(torch.isnan(y_train).any())

torch.float32
torch.float32
tensor(False)
tensor(False)


In [6]:
cat ConvTran/Models/model.py

import numpy as np
from torch import nn
from Models.AbsolutePositionalEncoding import tAPE, AbsolutePositionalEncoding, LearnablePositionalEncoding
from Models.Attention import Attention, Attention_Rel_Scl, Attention_Rel_Vec


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class Permute(nn.Module):
    def forward(self, x):
        return x.permute(1, 0, 2)


def model_factory(config):
    if config['Net_Type'][0] == 'T':
        model = Transformer(config, output_size=config['output_size'])
    elif config['Net_Type'][0] == 'CC-T':
        model = CasualConvTran(config, output_size=config['output_size'])
    else:
        model = ConvTran(config, output_size=config['output_size'])
    return model


class Transformer(nn.Module):
    def __init__(self, config, output_size):
        super().__init__()
        # Parameters Initialization -----------------------------------------------
        channel_size, seq_len = config['Data_sh

In [7]:
train_loader, val_loader, test_loader = create_dataloaders(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    batch_size=64
)

X_batch, y_batch = next(iter(train_loader))

print("X:", X_batch.shape, X_batch.dtype)
print("y:", y_batch.shape, y_batch.dtype)

X: torch.Size([64, 1, 1000]) torch.float32
y: torch.Size([64, 2]) torch.float32


In [8]:
config = {
    # Input
    'Data_shape': (32, 1, 1000),

    # ConvTran architecture
    'emb_size': 16,
    'num_heads': 8,
    'dim_ff': 256,

    # Positional encoding
    'Fix_pos_encode': 'tAPE',
    'Rel_pos_encode': 'eRPE',

    # Dropout
    'dropout': 0.01,

    # Regression
    'output_size': 2,

    # Model type
    'Net_Type': ['C-T'],
}

In [9]:
import sys

sys.path.insert(
    0,
    "/data1/yashvi_bhuva/BP_estimation_using_PPG/ConvTran/ConvTran"
)
from Models.model import model_factory
model = model_factory(config)

X_batch, y_batch = next(iter(train_loader))

output = model(X_batch)

print("Input :", X_batch.shape)
print("Output:", output.shape)

/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4215.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/torch/nn/modules/conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/Convolution.cpp:1101.)
  return F.conv2d(


Input : torch.Size([64, 1, 1000])
Output: torch.Size([64, 2])


In [10]:
import torch.nn as nn

criterion = nn.SmoothL1Loss()

loss = criterion(output, y_batch)

print("Prediction shape:", output.shape)
print("Target shape:", y_batch.shape)
print("Loss:", loss.item())

loss.backward()

print("Backward pass successful")

Prediction shape: torch.Size([64, 2])
Target shape: torch.Size([64, 2])
Loss: 94.82847595214844
Backward pass successful


In [11]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

optimizer.step()

print("Optimizer step successful")

Optimizer step successful


In [12]:
model.train()

X_batch, y_batch = next(iter(train_loader))

optimizer.zero_grad()

pred = model(X_batch)

loss = criterion(pred, y_batch)

loss.backward()

optimizer.step()

print("Loss:", loss.item())
print("Training step successful")

Loss: 98.6328125
Training step successful


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model_factory(config).to(device)

In [14]:
loss_module = torch.nn.SmoothL1Loss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [15]:

from Training import SupervisedTrainer, validate, train_runner
trainer = SupervisedTrainer(
    model=model,
    dataloader=train_loader,
    device=device,
    loss_module=loss_module,
    optimizer=optimizer,
    l2_reg=None
)

val_evaluator = SupervisedTrainer(
    model=model,
    dataloader=val_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None
)

In [16]:
import os

config['epochs'] = 5
config['optimizer'] = optimizer
config['loss_module'] = loss_module
config['key_metric'] = 'loss'
config['save_dir'] = './checkpoints'

os.makedirs(config['save_dir'], exist_ok=True)

In [17]:
# metrics = trainer.train_epoch(1)

# print(metrics)

In [18]:
# val_metrics, results = val_evaluator.evaluate(1)

# print(val_metrics)

In [19]:
config['epochs'] = 5

train_runner(
    config=config,
    model=model,
    trainer=trainer,
    val_evaluator=val_evaluator,
    path='./checkpoints/best_model.pth'
)

Training Epoch:   0%|          | 0/5 [00:00<?, ?it/s]2026-09-21 20:51:35,968 | INFO : Validation Summary: epoch: 1.000000 | loss: 56.016767 | SBP_MAE: 86.490936 | DBP_MAE: 26.542614 | SBP_RMSE: 89.406685 | DBP_RMSE: 28.890253 | 



Best validation loss: 56.01676747354429
Saving best model for epoch: 1



2026-09-21 20:51:36,216 | INFO : Epoch 1 Training Summary: epoch: 1.000000 | loss: 79.454654 | SBP_MAE: 110.699905 | DBP_MAE: 49.209377 | SBP_RMSE: 113.544647 | DBP_RMSE: 51.798233 | 
Training Epoch:  20%|██        | 1/5 [15:12<1:00:48, 912.19s/it]2026-09-21 21:06:41,664 | INFO : Validation Summary: epoch: 2.000000 | loss: 13.555993 | SBP_MAE: 19.620012 | DBP_MAE: 8.474933 | SBP_RMSE: 25.381020 | DBP_RMSE: 11.491362 | 
2026-09-21 21:06:41,850 | INFO : Epoch 2 Training Summary: epoch: 2.000000 | loss: 29.623299 | SBP_MAE: 49.459824 | DBP_MAE: 10.774364 | SBP_RMSE: 57.905956 | DBP_RMSE: 14.638174 | 
Training Epoch:  40%|████      | 2/5 [30:17<45:25, 908.33s/it]  


Best validation loss: 13.555993435572926
Saving best model for epoch: 2



2026-09-21 21:21:47,847 | INFO : Validation Summary: epoch: 3.000000 | loss: 11.312654 | SBP_MAE: 15.908660 | DBP_MAE: 7.694540 | SBP_RMSE: 20.246622 | DBP_RMSE: 10.715707 | 
2026-09-21 21:21:47,981 | INFO : Epoch 3 Training Summary: epoch: 3.000000 | loss: 11.960614 | SBP_MAE: 17.020668 | DBP_MAE: 7.879127 | SBP_RMSE: 21.721367 | DBP_RMSE: 10.910276 | 
Training Epoch:  60%|██████    | 3/5 [45:23<30:14, 907.33s/it]


Best validation loss: 11.312654009336754
Saving best model for epoch: 3



2026-09-21 21:36:54,221 | INFO : Validation Summary: epoch: 4.000000 | loss: 10.876866 | SBP_MAE: 15.235058 | DBP_MAE: 7.493758 | SBP_RMSE: 19.603155 | DBP_RMSE: 10.551210 | 
2026-09-21 21:36:54,399 | INFO : Epoch 4 Training Summary: epoch: 4.000000 | loss: 10.806180 | SBP_MAE: 15.141112 | DBP_MAE: 7.446937 | SBP_RMSE: 19.465290 | DBP_RMSE: 10.474478 | 
Training Epoch:  80%|████████  | 4/5 [1:00:30<15:06, 906.97s/it]


Best validation loss: 10.876865732341708
Saving best model for epoch: 4



2026-09-21 21:52:01,289 | INFO : Validation Summary: epoch: 5.000000 | loss: 10.585014 | SBP_MAE: 14.787025 | DBP_MAE: 7.358042 | SBP_RMSE: 19.134142 | DBP_RMSE: 10.309216 | 
2026-09-21 21:52:01,396 | INFO : Epoch 5 Training Summary: epoch: 5.000000 | loss: 10.386776 | SBP_MAE: 14.530147 | DBP_MAE: 7.217731 | SBP_RMSE: 18.853596 | DBP_RMSE: 10.222569 | 
2026-09-21 21:52:01,399 | INFO : Train Time: 1.0 hours, 15.0 minutes, 37.37366700172424 seconds




Best validation loss: 10.585014099472554
Saving best model for epoch: 5



In [20]:
checkpoint = torch.load(
    './checkpoints/model_best.pth',
    map_location=device
)

model.load_state_dict(checkpoint['state_dict'])
model.to(device)

ConvTran(
  (embed_layer): Sequential(
    (0): Conv2d(1, 64, kernel_size=(1, 8), stride=(1, 1), padding=same)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): GELU(approximate='none')
  )
  (embed_layer2): Sequential(
    (0): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1), padding=valid)
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): GELU(approximate='none')
  )
  (Fix_Position): tAPE(
    (dropout): Dropout(p=0.01, inplace=False)
  )
  (attention_layer): Attention_Rel_Scl(
    (key): Linear(in_features=16, out_features=16, bias=False)
    (value): Linear(in_features=16, out_features=16, bias=False)
    (query): Linear(in_features=16, out_features=16, bias=False)
    (dropout): Dropout(p=0.01, inplace=False)
    (to_out): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (LayerNorm): LayerNorm((16,), eps=1e-05, elementwise_affine=True, bias=T

In [ ]:
test_evaluator = SupervisedTrainer(
    model=model,
    dataloader=test_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None,
    print_interval=10,
    console=True,
    print_conf_mat=False
)

In [ ]:
test_metrics, test_results = test_evaluator.evaluate(
    epoch_num=None,
    keep_all=True
)

In [ ]:
print("Test Results:")

for key, value in test_metrics.items():
    print(f"{key}: {value}")

Test Results:
epoch: None
loss: 10.846226478724883
SBP_MAE: 15.176384925842285
DBP_MAE: 7.4926347732543945
SBP_RMSE: 19.67003631591797
DBP_RMSE: 10.207221031188965


In [ ]:
y_true = test_results['targets']
y_pred = test_results['predictions']

print("True shape:", y_true.shape)
print("Pred shape:", y_pred.shape)

True shape: (66296, 2)
Pred shape: (66296, 2)


In [ ]:
import numpy as np
sbp_true = y_true[:, 0]
sbp_pred = y_pred[:, 0]

dbp_true = y_true[:, 1]
dbp_pred = y_pred[:, 1]

sbp_error = sbp_pred - sbp_true
dbp_error = dbp_pred - dbp_true

print("\n===== TEST RESULTS =====")

print(f"SBP MAE  : {np.mean(np.abs(sbp_error)):.2f} mmHg")
print(f"SBP RMSE : {np.sqrt(np.mean(sbp_error**2)):.2f} mmHg")
print(f"SBP ME   : {np.mean(sbp_error):.2f} mmHg")
print(f"SBP STD  : {np.std(sbp_error):.2f} mmHg")

print(f"\nDBP MAE  : {np.mean(np.abs(dbp_error)):.2f} mmHg")
print(f"DBP RMSE : {np.sqrt(np.mean(dbp_error**2)):.2f} mmHg")
print(f"DBP ME   : {np.mean(dbp_error):.2f} mmHg")
print(f"DBP STD  : {np.std(dbp_error):.2f} mmHg")


===== TEST RESULTS =====
SBP MAE  : 15.18 mmHg
SBP RMSE : 19.67 mmHg
SBP ME   : -3.63 mmHg
SBP STD  : 19.33 mmHg

DBP MAE  : 7.49 mmHg
DBP RMSE : 10.21 mmHg
DBP ME   : -1.52 mmHg
DBP STD  : 10.09 mmHg


In [26]:
config['epochs'] = 50

train_runner(
    config=config,
    model=model,
    trainer=trainer,
    val_evaluator=val_evaluator,
    path='./checkpoints/best_model.pth'
)

Training Epoch:   0%|          | 0/50 [00:00<?, ?it/s]2026-09-21 22:08:03,339 | INFO : Validation Summary: epoch: 1.000000 | loss: 10.367344 | SBP_MAE: 14.501851 | DBP_MAE: 7.207144 | SBP_RMSE: 18.783304 | DBP_RMSE: 10.157261 | 
2026-09-21 22:08:03,424 | INFO : Epoch 1 Training Summary: epoch: 1.000000 | loss: 10.136297 | SBP_MAE: 14.163980 | DBP_MAE: 7.082231 | SBP_RMSE: 18.503647 | DBP_RMSE: 10.075534 | 
Training Epoch:   2%|▏         | 1/50 [15:07<12:20:44, 907.02s/it]


Best validation loss: 10.367344256461406
Saving best model for epoch: 1



2026-09-21 22:26:33,845 | INFO : Validation Summary: epoch: 2.000000 | loss: 10.430273 | SBP_MAE: 14.699476 | DBP_MAE: 7.136159 | SBP_RMSE: 18.808926 | DBP_RMSE: 9.936096 | 
2026-09-21 22:26:33,850 | INFO : Epoch 2 Training Summary: epoch: 2.000000 | loss: 9.940126 | SBP_MAE: 13.877467 | DBP_MAE: 6.975713 | SBP_RMSE: 18.228952 | DBP_RMSE: 9.977684 | 
Training Epoch:   4%|▍         | 2/50 [33:37<13:41:20, 1026.67s/it]

2026-09-21 22:47:35,924 | INFO : Validation Summary: epoch: 3.000000 | loss: 10.066236 | SBP_MAE: 13.986481 | DBP_MAE: 7.119418 | SBP_RMSE: 18.289162 | DBP_RMSE: 10.060224 | 
2026-09-21 22:47:36,013 | INFO : Epoch 3 Training Summary: epoch: 3.000000 | loss: 9.777123 | SBP_MAE: 13.636689 | DBP_MAE: 6.889860 | SBP_RMSE: 17.996538 | DBP_RMSE: 9.894972 | 
Training Epoch:   6%|▌         | 3/50 [54:39<14:48:27, 1134.20s/it]


Best validation loss: 10.066235537233773
Saving best model for epoch: 3



...

2026-09-22 09:55:18,979 | INFO : Validation Summary: epoch: 47.000000 | loss: 8.992151 | SBP_MAE: 12.543203 | DBP_MAE: 6.409297 | SBP_RMSE: 16.889109 | DBP_RMSE: 9.348032 | 
2026-09-22 09:55:19,164 | INFO : Epoch 47 Training Summary: epoch: 47.000000 | loss: 7.995405 | SBP_MAE: 11.065166 | DBP_MAE: 5.888399 | SBP_RMSE: 15.439010 | DBP_RMSE: 8.921385 | 
Training Epoch:  94%|█████████▍| 47/50 [12:02:22<45:00, 900.23s/it]  


Best validation loss: 8.992151245094911
Saving best model for epoch: 47



2026-09-22 10:10:19,200 | INFO : Validation Summary: epoch: 48.000000 | loss: 9.384725 | SBP_MAE: 12.999620 | DBP_MAE: 6.742157 | SBP_RMSE: 17.137028 | DBP_RMSE: 9.453041 | 
2026-09-22 10:10:19,204 | INFO : Epoch 48 Training Summary: epoch: 48.000000 | loss: 7.971297 | SBP_MAE: 11.031693 | DBP_MAE: 5.873538 | SBP_RMSE: 15.403641 | DBP_RMSE: 8.906028 | 
Training Epoch:  96%|█████████▌| 48/50 [12:17:22<30:00, 900.17s/it]

2026-09-22 10:25:19,136 | INFO : Validation Summary: epoch: 49.000000 | loss: 9.043579 | SBP_MAE: 12.628091 | DBP_MAE: 6.426498 | SBP_RMSE: 16.890852 | DBP_RMSE: 9.501957 | 
2026-09-22 10:25:19,140 | INFO : Epoch 49 Training Summary: epoch: 49.000000 | loss: 7.955747 | SBP_MAE: 11.012597 | DBP_MAE: 5.861423 | SBP_RMSE: 15.380779 | DBP_RMSE: 8.896097 | 
Training Epoch:  98%|█████████▊| 49/50 [12:32:22<15:00, 900.10s/it]

2026-09-22 10:40:19,208 | INFO : Validation Summary: epoch: 50.000000 | loss: 9.153805 | SBP_MAE: 12.784885 | DBP_MAE: 6.492358 | SBP_RMSE: 17.019405 | DBP_RMSE: 9.411541 | 
2026-09-22 10:40:19,212 | INFO : Epoch 50 Training Summary: epoch: 50.000000 | loss: 7.944348 | SBP_MAE: 10.994959 | DBP_MAE: 5.856177 | SBP_RMSE: 15.361693 | DBP_RMSE: 8.891041 | 
2026-09-22 10:40:19,217 | INFO : Train Time: 12.0 hours, 47.0 minutes, 22.816583156585693 seconds



In [27]:
checkpoint = torch.load(
    './checkpoints/model_best.pth',
    map_location=device
)

model.load_state_dict(checkpoint['state_dict'])
model.to(device)

test_evaluator = SupervisedTrainer(
    model=model,
    dataloader=test_loader,
    device=device,
    loss_module=loss_module,
    optimizer=None,
    l2_reg=None,
    print_interval=10,
    console=True,
    print_conf_mat=False
)

In [28]:
test_metrics, test_results = test_evaluator.evaluate(
    epoch_num=None,
    keep_all=True
)

In [29]:
print("Test Results:")

for key, value in test_metrics.items():
    print(f"{key}: {value}")

Test Results:
epoch: None
loss: 9.244501983556958
SBP_MAE: 12.909143447875977
DBP_MAE: 6.550126075744629
SBP_RMSE: 17.687442779541016
DBP_RMSE: 9.341608047485352


In [30]:
y_true = test_results['targets']
y_pred = test_results['predictions']

print("True shape:", y_true.shape)
print("Pred shape:", y_pred.shape)

True shape: (66296, 2)
Pred shape: (66296, 2)


In [31]:
sbp_true = y_true[:, 0]
sbp_pred = y_pred[:, 0]

dbp_true = y_true[:, 1]
dbp_pred = y_pred[:, 1]

sbp_error = sbp_pred - sbp_true
dbp_error = dbp_pred - dbp_true
print("\n===== TEST RESULTS =====")

print(f"SBP MAE  : {np.mean(np.abs(sbp_error)):.2f} mmHg")
print(f"SBP RMSE : {np.sqrt(np.mean(sbp_error**2)):.2f} mmHg")
print(f"SBP ME   : {np.mean(sbp_error):.2f} mmHg")
print(f"SBP STD  : {np.std(sbp_error):.2f} mmHg")

print(f"\nDBP MAE  : {np.mean(np.abs(dbp_error)):.2f} mmHg")
print(f"DBP RMSE : {np.sqrt(np.mean(dbp_error**2)):.2f} mmHg")
print(f"DBP ME   : {np.mean(dbp_error):.2f} mmHg")
print(f"DBP STD  : {np.std(dbp_error):.2f} mmHg")


===== TEST RESULTS =====
SBP MAE  : 12.91 mmHg
SBP RMSE : 17.69 mmHg
SBP ME   : -2.39 mmHg
SBP STD  : 17.52 mmHg

DBP MAE  : 6.55 mmHg
DBP RMSE : 9.34 mmHg
DBP ME   : -0.28 mmHg
DBP STD  : 9.34 mmHg
